In [1]:
import random
from collections import defaultdict, Counter

# Zadanie 1

Zadanie polega na policzeniu częstolci występowania słów w angielskim tekście. Jakie słowa występują najczęściej i jaki procent wszystkich słów stanowią? Podobno przeciętny Polak zna 30 tys. słów, a posługuje się tylko 20% z tego zbioru, co daje tylko 6 tys. słów. Czy w Polsce panuje ubóstwo językowe? Sprawdz jaki procent wiedzy z Wikipedii umiałby przekazać przeciętny Polak, gdyby jego elokwencje przełożyć na grunt języka angielskiego. Policz jaki procent wszystkich słów stanowi zbiór 30 tys. najpopularniejszych słów, a jaki procent stanowi zbiór 6 tys..

In [2]:
with open("norm_wiki_sample.txt", "r") as f:
    wiki_text = f.read().lower()

words = wiki_text.split()
word_freq = {}

for word in words:
    if word in word_freq:
        word_freq[word] += 1
    else:
        word_freq[word] = 1

sorted_word_freq = sorted(word_freq.items(), key=lambda item: item[1], reverse=True)

total_words_count = len(words)
unique_words_count = len(word_freq)

top_6k_words = sorted_word_freq[:6000]
sum_top_6k = sum(count for word, count in top_6k_words)
percentage_6k = (sum_top_6k / total_words_count) * 100 if total_words_count > 0 else 0

# Oblicz pokrycie dla 30,000 najpopularniejszych słów
top_30k_words = sorted_word_freq[:30000]
sum_top_30k = sum(count for word, count in top_30k_words)
percentage_30k = (sum_top_30k / total_words_count) * 100 if total_words_count > 0 else 0

# 4. Wyświetl wyniki
print(f"Całkowita liczba słów w tekście: {total_words_count}")
print(f"Liczba unikalnych słów: {unique_words_count}")
print("-" * 30)
print(f"Zbiór {len(top_6k_words)} najpopularniejszych słów stanowi {percentage_6k:.2f}% wszystkich słów w tekście.")
print(f"Zbiór {len(top_30k_words)} najpopularniejszych słów stanowi {percentage_30k:.2f}% wszystkich słów w tekście.")
print("-" * 30)
print("10 najczęstszych słów:")
for word, count in sorted_word_freq[:10]:
    print(f"{word} - {count}")

Całkowita liczba słów w tekście: 1840506
Liczba unikalnych słów: 100835
------------------------------
Zbiór 6000 najpopularniejszych słów stanowi 82.25% wszystkich słów w tekście.
Zbiór 30000 najpopularniejszych słów stanowi 94.72% wszystkich słów w tekście.
------------------------------
10 najczęstszych słów:
the - 118991
of - 59073
and - 48804
in - 47667
a - 36762
to - 33997
was - 19579
is - 16649
for - 14178
on - 13896


# Zadanie 2
Używając wyliczonych prawdopodobieństw w poprzednim zadaniu, wygeneruj ciąg słów - przybliżenie pierwszego rzędu.

In [3]:
probabilities_dict = {}
for word, count in sorted_word_freq:
    probabilities_dict[word] = count / total_words_count if total_words_count > 0 else 0

def first_row_generator(probabilities_dict):
    chars = list(probabilities_dict.keys())
    probabilities = list(probabilities_dict.values())

    while True:
        yield random.choices(chars, weights=probabilities)[0]

generator = first_row_generator(probabilities_dict)
first_row = [next(generator) for _ in range(200)]
print("\nPierwszy wiersz wygenerowany na podstawie rozkładu słów z wikipedii:")
print(" ".join(first_row))


Pierwszy wiersz wygenerowany na podstawie rozkładu słów z wikipedii:
is opposition time improve score napoli was and pop humans name known selther of cardiff vengeful wakamatsu succeeded s the found 40 alumni overhaul year 1626 coast their and schrazade heavy work preserving remained series called 1st assume overpriced superbi the and left for after team patrick home where he department per been of estonian west g vietnam author england recruited award was with in bornean piano is that holds washington from stubbed ray she venezuela census his and hu patients took the franz with living and year as that culture r among the district not on amp mathematics minister in cg13de between acehnese joined alive he maria responsibility war to 1871 general relay elephant of the summer battalion liberty from november the approximately name div and the for neoproterozoic 1952 varieties speed and 192 in longest russian support as in and was scriptstyle j com m director series officials just august m

# Zadanie 3
### Treść
Wygeneruj przybliżenie języka angielskiego na podstawie źródła Markowa pierwszego rzędu na słowach (tzn. prawdopodobieństwo następnego słowa zależy od jednego poprzedniego słowa). (3pt) 

Implementacja powinna opierać się na łańcuchu Markowa, nie zaś na metodzie zaproponowanej przez Shannona, polegającej na wyszukiwaniu słów w tekście. Słowa wygenerowane w ten drugi sposób mogą pochodzić z rozkładu odbiegającego od rzeczywistego prawdopodobie«stwa warunkowego. Zastanów się, dlaczego tak jest. Następnie zrób to samo dla źródła Markowa drugiego rzędu (tzn. prawdopodobieństwo następnego słowa zależy od dwóch poprzednich słów). (3pt) 

Na koniec wygeneruj przybliżenie źródła Markowa drugiego rzędu, zaczynając od słowa probability. (4pt)

In [ ]:
def read_text_file(file_name):
    with open(file_name, 'r') as f:
        return f.read().lower()
    
def calculate_word_frequencies(text, order):
    words = text.split()
    ngrams = defaultdict(Counter)

    for i in range(len(words) - order):
        context = tuple(words[i:i+order])
        next_word = words[i+order]
        ngrams[context][next_word] += 1
    return ngrams

def generate_markov_text(ngrams, order, length, start_context=None):
    if start_context is None:
        print("Nie podano kontekstu startowego. Wybieram losowy kontekst.")
        start_context = random.choice(list(ngrams.keys()))
    else:
        context_possible = [k for k in ngrams.keys() if k[0] == start_context] if start_context else list(ngrams.keys())
        if not context_possible:
            print("Nie można znaleźć podanego kontekstu startowego. Wybieram losowy kontekst.")
            start_context = random.choice(list(ngrams.keys()))
        else:
            start_context = random.choice(context_possible)

    current_context = start_context
    result = list(current_context)

    for _ in range(length):
        if current_context not in ngrams:
            break
        options = list(ngrams[current_context].keys())
        weights = list(ngrams[current_context].values())

        next_word = random.choices(options, weights=weights, k=1)[0]
        result.append(next_word)
        current_context = current_context[1:] + (next_word,)

    return " ".join(result)

Generator pierwszego rzędu

In [20]:
wiki_text = read_text_file("norm_wiki_sample.txt")
hamlet_text = read_text_file("norm_hamlet.txt")
romeo_text = read_text_file("norm_romeo.txt")

wiki_frequencies = calculate_word_frequencies(wiki_text, order=1)
hamlet_frequencies = calculate_word_frequencies(hamlet_text, order=1)
romeo_frequencies = calculate_word_frequencies(romeo_text, order=1)

print("\nWygenerowany tekst na podstawie Wikipedii:")
print(generate_markov_text(wiki_frequencies, order=1, length=200))

print("\nWygenerowany tekst na podstawie Hamleta:")
print(generate_markov_text(hamlet_frequencies, order=1, length=200))

print("\nWygenerowany tekst na podstawie Romea i Julii:")
print(generate_markov_text(romeo_frequencies, order=1, length=200))


Wygenerowany tekst na podstawie Wikipedii:
Nie można znaleźć podanego kontekstu startowego. Wybieram losowy kontekst.
250cc motocross track on it with the cause of solutions providers have changed with them independence of disrepair in october 2006 mesh of uri of revenue cutter all the number thirteen episodes however a 2005 06 charts throughout his refusal to various artists which allows them to grow sorghum canola alfalfa and country and there s opera fire suppression exchange can greatly feed the most probably best material to declare bankruptcy one votes the book of the south western greyhound jim powell lovelace ms gil as the british ballads opera composers juliane made during a fox liberty link to burnsall a commune are still living in semitic sentiments appeared in february 2007 bold p indicating that he became shrewsbury township in the rules of the americas division four teams reproducing system was in march 1963 it is frugally but this was recorded at the indonesian football

Generator drugiego rzędu, zaczynamy od słowa probability

In [21]:
wiki_frequencies_order_2 = calculate_word_frequencies(wiki_text, order=2)
hamlet_frequencies_order_2 = calculate_word_frequencies(hamlet_text, order=2)
romeo_frequencies_order_2 = calculate_word_frequencies(romeo_text, order=2)

print("\nWygenerowany tekst na podstawie Wikipedii (order=2):")
print(generate_markov_text(wiki_frequencies_order_2, order=2, length=200, start_context="probability"))

print("\nWygenerowany tekst na podstawie Hamleta (order=2):")
print(generate_markov_text(hamlet_frequencies_order_2, order=2, length=200, start_context="probability"))

print("\nWygenerowany tekst na podstawie Romea i Julii (order=2):")
print(generate_markov_text(romeo_frequencies_order_2, order=2, length=200, start_context="probability"))


Wygenerowany tekst na podstawie Wikipedii (order=2):
probability be an exceptional concentration of a house blend 50 percent of the drugs b successful delivery of digitized golfers the first game in the southern mitchell shire with the original owner after he had two sons many beautiful wreaths were placed into a singing career independent of one of the heich legend the river crossing it again in 1865 this former bad boy reed forced to retired from the baluchistan gazetteer that certain nausherwan proceeded with abdulla khan of bulgaria since that time in the 1880s and to recognize yuan zhao into the repertoire as well as participate in men taking supplements with high energy performances by various means in the u s based nba teams and also opened new mints in eastern france callsignmeaning kaaq 105 9 rail kilometers from the village had been converted into a formal or ritualized introduction or offering as with the ice receded reindeer grazed on the asiatic water buffalo b mindorensi